In [1]:
!pip install scikit-learn

In [3]:
import pandas as pd

df = pd.read_csv(r'C:\Users\HP\Downloads\fraud_risk_management\data\feature_engineered_transactions.csv')

df["transaction_date"] = pd.to_datetime(df["transaction_date"])

print(df.shape)

(550000, 26)


In [4]:
# y = the answer we want to predict
y = df["is_fraud"]

# X = everything else EXCEPT the columns we decided to drop (Bucket A from before)
X = df.drop(columns=[
    "is_fraud",           # the answer itself — must never be a clue
    "transaction_id",     # just a label, not a real pattern
    "customer_id",        # just a label, not a real pattern
    "transaction_date",   # already extracted useful info from this
    "transaction_time"    # already extracted useful info from this
])

print(X.shape)

(550000, 21)


In [5]:
X = pd.get_dummies(X, drop_first=True)

print(X.shape)

(550000, 66)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,      # 20% goes into the test pile, 80% for training
    random_state=42,    # makes the split reproducible — same split every time you run this
    stratify=y          # important for fraud data — explained below
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (440000, 66)
Test size: (110000, 66)


In [7]:
X.isnull().sum().sort_values(ascending=False).head(10)

customer_avg_amount        79916
transaction_amount             0
account_balance                0
credit_score                   0
emi_amount                     0
has_loan                       0
transaction_hour               0
exceeds_balance                0
amount_to_balance_ratio        0
amount_vs_customer_avg         0
dtype: int64

In [8]:
X = X.drop(columns=["customer_avg_amount"])

# confirm no NaNs remain anywhere
print(X.isnull().sum().sum())

0


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (440000, 65)
Test size: (110000, 65)


In [10]:
from sklearn.linear_model import LogisticRegression

# create the model (it doesn't know anything yet — this just sets it up)
model = LogisticRegression(max_iter=1000)

# this is the "learning" step — the model looks at X_train and y_train together,
# and figures out how much weight to give each column to predict fraud
model.fit(X_train, y_train)

print("Model trained!")

Model trained!


C:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
print("Model trained!")

Model trained!


In [13]:
from sklearn.metrics import classification_report, confusion_matrix

# ask the model to predict on the test set (data it has NEVER seen during training)
y_pred = model.predict(X_test_scaled)

# compare predictions vs actual answers
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[109018      7]
 [   975      0]]
              precision    recall  f1-score   support

           0       0.99      1.00      1.00    109025
           1       0.00      0.00      0.00       975

    accuracy                           0.99    110000
   macro avg       0.50      0.50      0.50    110000
weighted avg       0.98      0.99      0.99    110000



In [14]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[82570 26455]
 [  680   295]]
              precision    recall  f1-score   support

           0       0.99      0.76      0.86    109025
           1       0.01      0.30      0.02       975

    accuracy                           0.75    110000
   macro avg       0.50      0.53      0.44    110000
weighted avg       0.98      0.75      0.85    110000



In [15]:
from sklearn.ensemble import RandomForestClassifier

# class_weight='balanced' here too — same reasoning as before, fraud is rare so we tell it to care more about catching it
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

rf_model.fit(X_train_scaled, y_train)

print("Random Forest trained!")

Random Forest trained!


In [16]:
y_pred_rf = rf_model.predict(X_test_scaled)

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[109025      0]
 [   974      1]]
              precision    recall  f1-score   support

           0       0.99      1.00      1.00    109025
           1       1.00      0.00      0.00       975

    accuracy                           0.99    110000
   macro avg       1.00      0.50      0.50    110000
weighted avg       0.99      0.99      0.99    110000



In [17]:
# get the probability of fraud (not just a 0/1 guess) for each test transaction
y_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# look at the distribution of scores for actual fraud vs actual non-fraud
import pandas as pd
scores_df = pd.DataFrame({"actual": y_test, "fraud_probability": y_proba_rf})
print(scores_df.groupby("actual")["fraud_probability"].describe())

           count      mean       std  min   25%   50%   75%   max
actual                                                           
0       109025.0  0.022078  0.024561  0.0  0.01  0.02  0.03  0.50
1          975.0  0.035774  0.055834  0.0  0.01  0.02  0.03  0.63


In [18]:
# try a threshold of 0.10 instead of the default 0.50
threshold = 0.10
y_pred_threshold = (y_proba_rf >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_threshold))
print(classification_report(y_test, y_pred_threshold))

[[107696   1329]
 [   893     82]]
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    109025
           1       0.06      0.08      0.07       975

    accuracy                           0.98    110000
   macro avg       0.52      0.54      0.53    110000
weighted avg       0.98      0.98      0.98    110000



In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

for threshold in [0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]:
    y_pred_t = (y_proba_rf >= threshold).astype(int)
    
    precision = precision_score(y_test, y_pred_t)
    recall = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    
    print(f"Threshold {threshold}: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

Threshold 0.02: Precision=0.010, Recall=0.613, F1=0.019
Threshold 0.03: Precision=0.010, Recall=0.388, F1=0.020
Threshold 0.05: Precision=0.016, Recall=0.178, F1=0.029
Threshold 0.08: Precision=0.041, Recall=0.104, F1=0.059
Threshold 0.1: Precision=0.058, Recall=0.084, F1=0.069
Threshold 0.15: Precision=0.074, Recall=0.052, F1=0.061
Threshold 0.2: Precision=0.075, Recall=0.030, F1=0.043


In [20]:
!pip install xgboost

In [21]:
from xgboost import XGBClassifier

In [22]:
# scale_pos_weight tells XGBoost how much more to care about the rare class (fraud)
# a common formula: (number of legitimate cases) / (number of fraud cases)
scale = y_train.value_counts()[0] / y_train.value_counts()[1]

xgb_model = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale,
    random_state=42,
    eval_metric='logloss'
)

xgb_model.fit(X_train_scaled, y_train)

print("XGBoost trained!")

XGBoost trained!


In [23]:
y_proba_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]

# same threshold scan as before, this time on XGBoost's scores
for threshold in [0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]:
    y_pred_t = (y_proba_xgb >= threshold).astype(int)
    
    precision = precision_score(y_test, y_pred_t)
    recall = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    
    print(f"Threshold {threshold}: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

Threshold 0.02: Precision=0.009, Recall=0.985, F1=0.017
Threshold 0.03: Precision=0.009, Recall=0.978, F1=0.017
Threshold 0.05: Precision=0.009, Recall=0.959, F1=0.017
Threshold 0.08: Precision=0.009, Recall=0.920, F1=0.017
Threshold 0.1: Precision=0.009, Recall=0.903, F1=0.018
Threshold 0.15: Precision=0.009, Recall=0.836, F1=0.018
Threshold 0.2: Precision=0.009, Recall=0.768, F1=0.018


In [24]:
# what fraction of the ENTIRE test set gets flagged as "fraud" at threshold 0.20 (your strictest one)?
flagged = (y_proba_xgb >= 0.20).sum()
total = len(y_proba_xgb)
print(f"Flagged as fraud at threshold 0.20: {flagged} out of {total} ({flagged/total*100:.1f}%)")

# also look at the overall shape of the probability scores themselves
import pandas as pd
pd.Series(y_proba_xgb).describe()

Flagged as fraud at threshold 0.20: 81419 out of 110000 (74.0%)


count    110000.000000
mean          0.321990
std           0.165066
min           0.000053
25%           0.194704
50%           0.318292
75%           0.439498
max           0.985269
dtype: float64

In [25]:
# instead of the full ratio (~110), try something more moderate
xgb_model2 = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=10,  # much gentler than ~110
    random_state=42,
    eval_metric='logloss'
)

xgb_model2.fit(X_train_scaled, y_train)

y_proba_xgb2 = xgb_model2.predict_proba(X_test_scaled)[:, 1]

for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred_t = (y_proba_xgb2 >= threshold).astype(int)
    precision = precision_score(y_test, y_pred_t)
    recall = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    print(f"Threshold {threshold}: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

Threshold 0.3: Precision=0.048, Recall=0.054, F1=0.051
Threshold 0.4: Precision=0.065, Recall=0.038, F1=0.048
Threshold 0.5: Precision=0.058, Recall=0.016, F1=0.026
Threshold 0.6: Precision=0.053, Recall=0.005, F1=0.009
Threshold 0.7: Precision=0.091, Recall=0.003, F1=0.006


In [26]:
xgb_model3 = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=30,  # between 10 (too weak) and 110 (too strong)
    random_state=42,
    eval_metric='logloss'
)

xgb_model3.fit(X_train_scaled, y_train)

y_proba_xgb3 = xgb_model3.predict_proba(X_test_scaled)[:, 1]

for threshold in [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4]:
    y_pred_t = (y_proba_xgb3 >= threshold).astype(int)
    precision = precision_score(y_test, y_pred_t)
    recall = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    print(f"Threshold {threshold}: Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")

Threshold 0.1: Precision=0.009, Recall=0.674, F1=0.018
Threshold 0.15: Precision=0.010, Recall=0.471, F1=0.020
Threshold 0.2: Precision=0.012, Recall=0.287, F1=0.023
Threshold 0.25: Precision=0.015, Recall=0.182, F1=0.028
Threshold 0.3: Precision=0.021, Recall=0.122, F1=0.035
Threshold 0.35: Precision=0.028, Recall=0.089, F1=0.042
Threshold 0.4: Precision=0.036, Recall=0.074, F1=0.048


In [27]:
flagged_xgb3 = (y_proba_xgb3 >= 0.10).sum()
print(f"Flagged as fraud at threshold 0.10: {flagged_xgb3} out of {110000} ({flagged_xgb3/110000*100:.1f}%)")

Flagged as fraud at threshold 0.10: 70566 out of 110000 (64.2%)


In [28]:
flagged_rf = (y_proba_rf >= 0.02).sum()
print(f"Random Forest flagged at 0.02: {flagged_rf} out of 110000 ({flagged_rf/110000*100:.1f}%)")

Random Forest flagged at 0.02: 61140 out of 110000 (55.6%)


In [29]:
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    y_pred_t = (y_proba_rf >= threshold).astype(int)
    flagged_pct = y_pred_t.sum() / len(y_pred_t) * 100
    
    precision = precision_score(y_test, y_pred_t, zero_division=0)
    recall = recall_score(y_test, y_pred_t, zero_division=0)
    
    print(f"Threshold {threshold}: Flagged={flagged_pct:.1f}%, Precision={precision:.3f}, Recall={recall:.3f}")

Threshold 0.3: Flagged=0.1%, Precision=0.118, Recall=0.011
Threshold 0.4: Flagged=0.0%, Precision=0.154, Recall=0.002
Threshold 0.5: Flagged=0.0%, Precision=0.500, Recall=0.001
Threshold 0.6: Flagged=0.0%, Precision=1.000, Recall=0.001
Threshold 0.7: Flagged=0.0%, Precision=0.000, Recall=0.000
Threshold 0.8: Flagged=0.0%, Precision=0.000, Recall=0.000


In [30]:
for threshold in [0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18, 0.20, 0.22, 0.25, 0.28]:
    y_pred_t = (y_proba_rf >= threshold).astype(int)
    flagged_pct = y_pred_t.sum() / len(y_pred_t) * 100
    
    precision = precision_score(y_test, y_pred_t, zero_division=0)
    recall = recall_score(y_test, y_pred_t, zero_division=0)
    
    print(f"Threshold {threshold}: Flagged={flagged_pct:.2f}%, Precision={precision:.3f}, Recall={recall:.3f}")

Threshold 0.02: Flagged=55.58%, Precision=0.010, Recall=0.613
Threshold 0.04: Flagged=18.37%, Precision=0.012, Recall=0.249
Threshold 0.06: Flagged=5.61%, Precision=0.022, Recall=0.138
Threshold 0.08: Flagged=2.25%, Precision=0.041, Recall=0.104
Threshold 0.1: Flagged=1.28%, Precision=0.058, Recall=0.084
Threshold 0.12: Flagged=0.91%, Precision=0.071, Recall=0.073
Threshold 0.14: Flagged=0.72%, Precision=0.072, Recall=0.058
Threshold 0.16: Flagged=0.56%, Precision=0.070, Recall=0.044
Threshold 0.18: Flagged=0.45%, Precision=0.073, Recall=0.037
Threshold 0.2: Flagged=0.35%, Precision=0.075, Recall=0.030
Threshold 0.22: Flagged=0.26%, Precision=0.074, Recall=0.022
Threshold 0.25: Flagged=0.18%, Precision=0.071, Recall=0.014
Threshold 0.28: Flagged=0.12%, Precision=0.092, Recall=0.012


In [31]:
import pandas as pd

# build a small results table combining actual answer + model's probability score
results = pd.DataFrame({
    "actual_fraud": y_test.values,
    "fraud_probability": y_proba_rf
})

# define the band boundaries
def assign_risk_band(prob):
    if prob >= 0.40:
        return "Critical"
    elif prob >= 0.25:
        return "High"
    elif prob >= 0.10:
        return "Medium"
    else:
        return "Low"

results["risk_band"] = results["fraud_probability"].apply(assign_risk_band)

results.head(10)

,actual_fraud,fraud_probability,risk_band
0,0,0.04,Low
1,0,0.01,Low
2,0,0.02,Low
3,0,0.00,Low
4,0,0.02,Low
5,0,0.00,Low
6,0,0.00,Low
7,0,0.03,Low
8,0,0.01,Low
9,0,0.06,Low


In [32]:
# how many transactions fall into each band, and what's the REAL fraud rate within each band?
band_summary = results.groupby("risk_band").agg(
    total_transactions=("actual_fraud", "count"),
    actual_fraud_count=("actual_fraud", "sum"),
    fraud_rate_pct=("actual_fraud", lambda x: x.mean() * 100)
)

print(band_summary)

           total_transactions  actual_fraud_count  fraud_rate_pct
risk_band                                                        
Critical                   13                   2       15.384615
High                      184                  12        6.521739
Low                    108589                 893        0.822367
Medium                   1214                  68        5.601318


In [33]:
# sort by fraud_probability, highest risk first — this IS the investigation priority order
priority_list = results.sort_values(by="fraud_probability", ascending=False)

# add a rank number so it's clear this is a queue to work through, top to bottom
priority_list["investigation_rank"] = range(1, len(priority_list) + 1)

# reorder columns so rank comes first, most useful for reading
priority_list = priority_list[["investigation_rank", "fraud_probability", "risk_band", "actual_fraud"]]

priority_list.head(20)

,investigation_rank,fraud_probability,risk_band,actual_fraud
5833,1,0.63,Critical,1
101113,2,0.50,Critical,0
108648,3,0.47,Critical,0
95994,4,0.46,Critical,0
97168,5,0.46,Critical,0
88976,6,0.46,Critical,0
1286,7,0.44,Critical,0
27536,8,0.44,Critical,0
86437,9,0.43,Critical,0
101924,10,0.42,Critical,0


In [34]:
# get the transaction_id values for exactly the rows in your test set
# X_test.index gives you the original row positions that ended up in the test set after the split
test_transaction_ids = df.loc[X_test.index, "transaction_id"]

# add it into your results table
results["transaction_id"] = test_transaction_ids.values

priority_list = results.sort_values(by="fraud_probability", ascending=False)
priority_list["investigation_rank"] = range(1, len(priority_list) + 1)
priority_list = priority_list[["investigation_rank", "transaction_id", "fraud_probability", "risk_band", "actual_fraud"]]

priority_list.head(20)

,investigation_rank,transaction_id,fraud_probability,risk_band,actual_fraud
5833,1,TXN000077173,0.63,Critical,1
101113,2,TXN000409467,0.50,Critical,0
108648,3,TXN000190105,0.47,Critical,0
95994,4,TXN000010441,0.46,Critical,0
97168,5,TXN000204823,0.46,Critical,0
88976,6,TXN000309812,0.46,Critical,0
1286,7,TXN000022965,0.44,Critical,0
27536,8,TXN000068128,0.44,Critical,0
86437,9,TXN000279081,0.43,Critical,0
101924,10,TXN000419883,0.42,Critical,0


In [35]:
priority_list.to_csv("investigation_priority_list.csv", index=False)
print("Saved:", priority_list.shape)

Saved: (110000, 5)


In [36]:
# how much fraud VALUE sits in just the Critical + High bands? 
# (need to bring back transaction_amount the same way you did transaction_id)
results["transaction_amount"] = df.loc[X_test.index, "transaction_amount"].values

high_priority_value = results[results["risk_band"].isin(["Critical", "High"])]
print("Total transactions in Critical+High:", len(high_priority_value))
print("Actual fraud transactions among them:", high_priority_value["actual_fraud"].sum())
print("Fraud value captured (₹):", high_priority_value[high_priority_value["actual_fraud"]==1]["transaction_amount"].sum())

Total transactions in Critical+High: 197
Actual fraud transactions among them: 14
Fraud value captured (₹): 12826180.709999999


In [40]:
# apply the SAME one-hot encoding, same column order, same scaler — to the FULL dataset
X_full = df.drop(columns=["is_fraud", "transaction_id", "customer_id", "transaction_date", "transaction_time", "customer_avg_amount"])
X_full = pd.get_dummies(X_full, drop_first=True)

# make sure columns match exactly what the model was trained on (same order, same columns)
X_full = X_full.reindex(columns=X_train.columns, fill_value=0)

X_full_scaled = scaler.transform(X_full)

df["fraud_probability"] = rf_model.predict_proba(X_full_scaled)[:, 1]
df["risk_band"] = df["fraud_probability"].apply(assign_risk_band)

df["is_test_set"] = df.index.isin(X_test.index)

df.to_csv(r'C:\Users\HP\Downloads\fraud_risk_management\data\power_bi_final_data.csv', index=False)
print(df.shape)

(550000, 29)


In [41]:
import os
print(os.getcwd())

C:\Users\HP\Downloads\fraud_risk_management\notebooks


In [42]:
print(df.columns.tolist())
print("is_test_set" in df.columns)

['transaction_id', 'customer_id', 'transaction_date', 'transaction_time', 'account_type', 'transaction_type', 'transaction_amount', 'transaction_direction', 'account_balance', 'merchant_category', 'state', 'credit_score', 'has_loan', 'loan_type', 'emi_amount', 'transaction_status', 'channel', 'kyc_status', 'is_fraud', 'transaction_hour', 'exceeds_balance', 'amount_to_balance_ratio', 'customer_avg_amount', 'amount_vs_customer_avg', 'customer_transaction_count', 'days_since_last_transaction', 'fraud_probability', 'risk_band', 'is_test_set']
True


In [43]:
import os
print(os.path.abspath(r'C:\Users\HP\Downloads\fraud_risk_management\data\power_bi_final_data.csv'))

C:\Users\HP\Downloads\fraud_risk_management\data\power_bi_final_data.csv
